# Yoruba Intent Classifier — Whisper-Based Training (Drive Only)
This notebook loads everything from Google Drive — no manual uploads needed.

**Required files in Google Drive → Colab Notebooks:**
- `recordings.zip` — your voice recordings
- `whisper_yoruba_best_v4.pt` — fine-tuned Whisper model  
- `yoruba_commands.csv` — your cleaned CSV

**Before running:** Runtime → Change runtime type → T4 GPU


## Step 1 — Mount Google Drive

In [1]:
from google.colab import drive
import os, torch

drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/MyDrive/Colab Notebooks'

print("Files in Drive folder:")
for f in sorted(os.listdir(DRIVE_PATH)):
    size = os.path.getsize(os.path.join(DRIVE_PATH, f))
    print("  " + f + " (" + str(round(size/1024/1024, 1)) + " MB)")

print()
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Not available")

Mounted at /content/drive
Files in Drive folder:
  Untitled0.ipynb (0.0 MB)
  Untitled1.ipynb (0.0 MB)
  Untitled2.ipynb (0.0 MB)
  Untitled3.ipynb (0.0 MB)
  Untitled4.ipynb (0.0 MB)
  Yoruba_ASR_Training_v3 (3).ipynb (0.0 MB)
  Yoruba_Intent_Training (2).ipynb (0.0 MB)
  Yoruba_Intent_Whisper_Training_v2.ipynb (0.0 MB)
  Yoruba_TTS_Training_v2 (1).ipynb (0.0 MB)
  Yoruba_TTS_Training_v2.ipynb (0.0 MB)
  Yoruba_Wav2Vec2_Finetune.ipynb (0.0 MB)
  Yoruba_Whisper_Training_v4.ipynb (0.0 MB)
  conformer_yoruba.nemo (72.8 MB)
  dataset_audio.zip (49.6 MB)
  intent_model_v4.pkl (0.8 MB)
  intent_model_v5.pkl (0.8 MB)
  recordings.zip (52.1 MB)
  train.jsonl (1.1 MB)
  val.jsonl (0.2 MB)
  whisper_yoruba_best.pt (922.3 MB)
  whisper_yoruba_best_v4.pt (922.3 MB)
  yoruba_commands_v5.csv (0.0 MB)

GPU: Tesla T4


## Step 2 — Install dependencies

In [2]:
!pip install -q openai-whisper scikit-learn pandas
print("Done!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 6.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Done!


## Step 3 — Extract recordings from Drive

In [3]:
import zipfile, os

os.makedirs('/content/recordings', exist_ok=True)

zip_path = DRIVE_PATH + '/recordings.zip'

if not os.path.exists(zip_path):
    raise FileNotFoundError("recordings.zip not found in Drive! Make sure it is in: " + DRIVE_PATH)

print("Extracting recordings.zip from Drive...")
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/recordings/')

# Auto detect base folder
base = '/content/recordings'
contents = [c for c in os.listdir(base) if not c.startswith('.')]
print("Contents: " + str(contents[:5]))

# Handle nested folder (e.g. recordings/recordings/)
if len(contents) == 1 and os.path.isdir(os.path.join(base, contents[0])):
    base = os.path.join(base, contents[0])
    print("Nested folder detected, using: " + base)

intents   = [d for d in os.listdir(base) if os.path.isdir(os.path.join(base, d))]
wav_files = []
for intent in intents:
    for f in os.listdir(os.path.join(base, intent)):
        if f.endswith('.wav'):
            wav_files.append((os.path.join(base, intent, f), intent))

print("Intent folders : " + str(len(intents)))
print("Total WAV files: " + str(len(wav_files)))
print()
print("Intents:")
from collections import Counter
counts = Counter(intent for _, intent in wav_files)
for intent, count in sorted(counts.items()):
    print("  " + intent + ": " + str(count))

Extracting recordings.zip from Drive...
Contents: ['recordings']
Nested folder detected, using: /content/recordings/recordings
Intent folders : 59
Total WAV files: 1398

Intents:
  apology: 15
  battery: 25
  bluetooth_off: 25
  bluetooth_on: 21
  brightness_down: 21
  brightness_up: 20
  calculator: 25
  camera: 25
  cancel: 25
  chrome: 25
  close_app: 25
  close_chrome: 25
  close_excel: 25
  close_notepad: 25
  close_tab: 25
  close_whatsapp: 25
  close_word: 25
  confirm: 21
  cpu: 25
  date: 25
  excel: 25
  exit: 30
  explorer: 25
  go_back: 25
  greeting: 51
  lock: 21
  music: 20
  mute: 20
  name: 24
  new_tab: 25
  next_song: 16
  notepad: 25
  pause_music: 25
  powerpoint: 25
  prev_song: 20
  ram: 25
  refresh: 21
  restart: 25
  screenshot: 25
  search: 25
  search_youtube: 25
  self_intro: 25
  settings: 11
  shutdown: 26
  sleep: 16
  storage: 15
  store: 25
  taskmanager: 25
  thanks: 30
  time: 25
  volume_down: 20
  volume_up: 16
  vscode: 25
  weather: 21
  what_can

In [10]:
# DEBUG - verify recordings loaded correctly
print("Base path:", base)
print("Total WAV files found:", len(wav_files))
print()

# Show first 5 files
for fpath, intent in wav_files[:5]:
    print(intent + " -> " + fpath)

# Show intent distribution
from collections import Counter
counts = Counter(intent for _, intent in wav_files)
print()
print("Intent distribution:")
for intent, count in sorted(counts.items()):
    print("  " + intent + ": " + str(count))

Base path: /content/recordings/recordings
Total WAV files found: 1398

prev_song -> /content/recordings/recordings/prev_song/orin_sẹhin_018.wav
prev_song -> /content/recordings/recordings/prev_song/previous_song_006.wav
prev_song -> /content/recordings/recordings/prev_song/orin_ti_o_koja_007.wav
prev_song -> /content/recordings/recordings/prev_song/go_back_song_017.wav
prev_song -> /content/recordings/recordings/prev_song/orin_ti_o_koja_005.wav

Intent distribution:
  apology: 15
  battery: 25
  bluetooth_off: 25
  bluetooth_on: 21
  brightness_down: 21
  brightness_up: 20
  calculator: 25
  camera: 25
  cancel: 25
  chrome: 25
  close_app: 25
  close_chrome: 25
  close_excel: 25
  close_notepad: 25
  close_tab: 25
  close_whatsapp: 25
  close_word: 25
  confirm: 21
  cpu: 25
  date: 25
  excel: 25
  exit: 30
  explorer: 25
  go_back: 25
  greeting: 51
  lock: 21
  music: 20
  mute: 20
  name: 24
  new_tab: 25
  next_song: 16
  notepad: 25
  pause_music: 25
  powerpoint: 25
  prev_song

## Step 4 — Load fine-tuned Whisper model from Drive

In [4]:
import whisper, torch

DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_PATH = DRIVE_PATH + '/whisper_yoruba_best_v4.pt'

print("Loading Whisper small...")
model = whisper.load_model('small').to(DEVICE)

if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    print("Fine-tuned Yoruba weights loaded!")
else:
    print("WARNING: whisper_yoruba_best_v4.pt not found in Drive")
    print("Using base Whisper model instead")
    print("Files available: " + str(os.listdir(DRIVE_PATH)))

model.eval()
print("Model ready on: " + DEVICE)

Loading Whisper small...


100%|███████████████████████████████████████| 461M/461M [00:05<00:00, 81.7MiB/s]


Fine-tuned Yoruba weights loaded!
Model ready on: cuda


## Step 5 — Transcribe all recordings with Whisper (GPU)

In [5]:
from tqdm import tqdm

transcriptions = []
failed         = 0

print("Transcribing " + str(len(wav_files)) + " files...")
print("Estimated time: ~10-15 minutes on GPU")
print()

for fpath, intent in tqdm(wav_files):
    try:
        result = model.transcribe(
            fpath,
            language='yo',
            fp16=(DEVICE == 'cuda'),
            temperature=0.0,
            no_speech_threshold=0.6,
            condition_on_previous_text=False,
        )
        text = result['text'].strip().lower()

        # Filter bad outputs
        words = text.split()
        if not text or len(words) == 0:
            failed += 1
            continue
        if len(words) > 15:
            failed += 1
            continue
        if len(words) > 4 and len(set(words)) == 1:
            failed += 1
            continue

        transcriptions.append({'text': text, 'intent': intent})

    except Exception as e:
        failed += 1

print()
print("Transcribed    : " + str(len(transcriptions)))
print("Failed/skipped : " + str(failed))
print()

# Show samples
from collections import defaultdict
by_intent = defaultdict(list)
for t in transcriptions:
    by_intent[t['intent']].append(t['text'])

print("=== SAMPLE TRANSCRIPTIONS ===")
for intent, texts in sorted(by_intent.items())[:8]:
    print(intent + ":")
    for t in texts[:2]:
        print("  -> " + t)

Transcribing 1398 files...
Estimated time: ~10-15 minutes on GPU



100%|██████████| 1398/1398 [08:15<00:00,  2.82it/s]


Transcribed    : 1360
Failed/skipped : 38

=== SAMPLE TRANSCRIPTIONS ===
apology:
  -> sorry
  -> binu
battery:
  -> battery level
  -> kini ipele batiri mi
bluetooth_off:
  -> bluetooth
  -> bluetooth duro
bluetooth_on:
  -> bluetooth sii
  -> on bluetooth
brightness_down:
  -> crease brightness
  -> imole
brightness_up:
  -> crease brightness
  -> crease brightness
calculator:
  -> isiro
  -> isiro
camera:
  -> kamera
  -> kamera


## Step 6 — Load cleaned CSV from Drive and combine with transcriptions

In [6]:
import pandas as pd

csv_path = DRIVE_PATH + '/yoruba_commands.csv'

if os.path.exists(csv_path):
    existing_df = pd.read_csv(csv_path)
    print("Loaded CSV: " + str(len(existing_df)) + " examples")
else:
    existing_df = pd.DataFrame(columns=['text', 'intent'])
    print("No CSV found in Drive - using transcriptions only")

# Build transcription dataframe
trans_df = pd.DataFrame(transcriptions)
print("Transcriptions: " + str(len(trans_df)) + " examples")

# Combine both
combined = pd.concat([existing_df[['text','intent']], trans_df[['text','intent']]], ignore_index=True)
combined = combined.dropna()
combined['text'] = combined['text'].str.strip().str.lower()
combined = combined.drop_duplicates(subset='text')

print("Combined total : " + str(len(combined)) + " examples")
print("Unique intents : " + str(combined['intent'].nunique()))
print()
print("Intent distribution:")
print(combined['intent'].value_counts().to_string())

No CSV found in Drive - using transcriptions only
Transcriptions: 1360 examples
Combined total : 395 examples
Unique intents : 59

Intent distribution:
intent
greeting           18
what_can_you_do    14
exit               12
thanks             12
close_app          11
self_intro         11
date               10
volume_down        10
music               9
restart             9
brightness_up       8
brightness_down     8
ram                 8
close_chrome        8
lock                8
cpu                 8
name                8
store               8
weather             7
excel               7
explorer            7
chrome              7
close_whatsapp      7
battery             7
prev_song           6
refresh             6
pause_music         6
shutdown            6
time                6
storage             6
apology             6
powerpoint          6
next_song           6
confirm             6
new_tab             6
bluetooth_off       6
volume_up           6
search              6
cance

## Step 7 — Train intent classifier

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
import joblib, numpy as np

texts  = combined['text'].tolist()
labels = combined['intent'].tolist()

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4))),
    ('clf',   LogisticRegression(max_iter=1000, C=5)),
])

scores = cross_val_score(pipeline, texts, labels, cv=5, scoring='accuracy')
print("CV accuracy: " + str(round(scores.mean()*100, 1)) + "% +/- " + str(round(scores.std()*100, 1)) + "%")

pipeline.fit(texts, labels)
joblib.dump(pipeline, '/content/intent_model_v5.pkl')
print("Intent model saved!")

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=5.
  warnings.warn(


CV accuracy: 71.1% +/- 3.8%
Intent model saved!


## Step 8 — Test on actual Whisper transcriptions

In [8]:
import random
random.shuffle(transcriptions)
test_sample = transcriptions[:20]

print("=== ACCURACY TEST ON WHISPER OUTPUT ===")
correct = 0
for t in test_sample:
    probs      = pipeline.predict_proba([t['text']])[0]
    intent     = pipeline.classes_[np.argmax(probs)]
    confidence = np.max(probs)
    status     = 'OK' if intent == t['intent'] else 'WRONG'
    if intent == t['intent']:
        correct += 1
    print(status + " | " + str(round(confidence*100)) + "% | '" + t['text'] + "' | expected=" + t['intent'] + " got=" + intent)

print()
print("Score: " + str(correct) + "/20 = " + str(round(correct/20*100)) + "%")

=== ACCURACY TEST ON WHISPER OUTPUT ===
OK | 59% | 'wifi run' | expected=wifi_off got=wifi_off
OK | 59% | 'je robot ni e' | expected=name got=name
OK | 34% | 'sẹhin' | expected=go_back got=go_back
OK | 72% | 'fe kamera' | expected=camera got=camera
OK | 38% | 'bluetooth sii' | expected=bluetooth_on got=bluetooth_on
OK | 76% | 'kini o le se' | expected=what_can_you_do got=what_can_you_do
OK | 60% | 'bluetooth' | expected=bluetooth_off got=bluetooth_off
OK | 39% | 'thank you' | expected=thanks got=thanks
OK | 23% | 'ṣeun ṣe' | expected=refresh got=refresh
OK | 66% | 'mo orin wa' | expected=music got=music
OK | 81% | 'tab titun' | expected=new_tab got=new_tab
WRONG | 28% | 'kọmputa' | expected=shutdown got=lock
OK | 43% | 'notepad' | expected=notepad got=notepad
OK | 39% | 'screen' | expected=lock got=lock
OK | 70% | 'open store' | expected=store got=store
OK | 28% | 'kọmputa' | expected=lock got=lock
OK | 58% | 'wa fun mi' | expected=search got=search
OK | 48% | 'settings' | expected=set

## Step 9 — Save both models to Google Drive

In [9]:
import shutil

# Save intent model
shutil.copy('/content/intent_model_v5.pkl', DRIVE_PATH + '/intent_model_v5.pkl')
print("Intent model saved: intent_model_v5.pkl")

# Save updated CSV
combined[['text','intent']].to_csv(DRIVE_PATH + '/yoruba_commands_v5.csv', index=False)
print("Updated CSV saved: yoruba_commands_v5.csv")

print()
print("=== DOWNLOAD AND REPLACE IN YOUR PROJECT ===")
print("intent_model_v5.pkl    -> models/intent_model.pkl")
print("yoruba_commands_v5.csv -> dataset/yoruba_commands.csv")

Intent model saved: intent_model_v5.pkl
Updated CSV saved: yoruba_commands_v5.csv

=== DOWNLOAD AND REPLACE IN YOUR PROJECT ===
intent_model_v5.pkl    -> models/intent_model.pkl
yoruba_commands_v5.csv -> dataset/yoruba_commands.csv
